### This notebook compute "P8. Soil susceptibility to water erosion" indicator for the 27 basins of IKI Project

Spanish: Susceptibilidad del suelo a la erosion hidrica

**Created:** 11/2025 by Sophia Bakar (sbakar@rti.org)  

**Project #:** 0219481  

**Last modified:** 1/2/2025 by Sophia (sbakar@rti.org)

**Status:** Complete for baseline and first future scenario.

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Exposicion\Scripts_Exposure
 
**Objective:** 

**Compatibility:** 

**Packages:** numpy, pandas, geopandas, sqlite3, rasterstats, rasterio, mapclassify

**Further documentation:**   

**Inputs:**   Peru AHD with districts

**Institutions** 

**Open acess or request data?** 

**Assumptions:** assumptions on script operations (e.g., inputs in subfolder of wd) or on calculations or theories used in code
 
**N/A Handling:** No missing data; skips over months or years with 0 mm precipitation when calculating R.  

**Future work:** 
 
**Notes:**   
General methodology:  
1. Compute erosivity (R) per COMID for each year in the meteorological data record.  
2. Compute the mean R across all years for each COMID.  
3. Read in rasters containing other parameters (K, C, P, and LS) used in the Revised Universal Soil Loss Equation (RUSLE) and compute the mean value for each COMID.  
4. Comptute mean soil loss (A) per COMID using the RUSLE equation, where A_mean = R_mean * K_mean * C_mean * LS_mean * P_mean.  
5. Categorize A_mean for each COMID into 4 categories (low, medium, high, very high, respectively indicated by 1, 2, 3, and 4) using natural breaks in the data.  

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
import geopandas as gpd
from rasterstats import zonal_stats
import rasterio
import mapclassify

In [ ]:
# set up user and database path
#user = 'jmayo'
#user= 'sgilson'
user = 'nreynolds'
db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
# db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"

In [ ]:
# set up indicator ID and get scenarios from database
IndID= 108 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

# For now: only baseline and first future
scenario_ids = scenarios_df.loc[
    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
].tolist()

# for all scenarios:
# scenario_ids = scenarios_df['ScnID'].tolist()

# get met source IDs for each scenario (we could modify this to do it dyanimically by querying the DB)
scenario_met_sources = {
    1: 2,  # Baseline (PISCO)
    2: 5,  # Future (CMIP6 85)
}

In [ ]:
subbasins_shapefile = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
# subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [ ]:
# calculate R (rainfall erositivity factor) - same as P3
precip_records = []   # store all COMID-year results
# conversion for cm to mm
cm_to_mm = 10.0

for grupo in range(1, 13):

    sqlite_path = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/Grupo_{grupo}/BD/BD_Grupo_{grupo}.sqlite'
    # sqlite_path = f"C:/Users/sbakar/OneDrive - Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/Grupo_{grupo}/BD/BD_Grupo_{grupo}.sqlite"
    conn = sqlite3.connect(sqlite_path)

    for ScnID in scenario_ids:
        met_source_id = scenario_met_sources.get(ScnID)
        if met_source_id is None:
            continue

        query = f"""
        SELECT
            comid,
            avg_precip_cm,
            measured_date
        FROM catchment_met_observations
        WHERE met_source_id = {met_source_id};
        """

        met = pd.read_sql_query(query, conn)

        if met.empty:
            continue

        met['measured_date'] = pd.to_datetime(
            met['measured_date'],
            format='%Y-%m-%d %H:%M:%S %z UTC',
            errors='coerce'
        )
        met = met.dropna(subset=['measured_date'])

        met['precip_mm'] = met['avg_precip_cm'] * cm_to_mm
        met['year'] = met['measured_date'].dt.year
        met['month'] = met['measured_date'].dt.month
        met['ScnID'] = ScnID

        # Monthly totals
        monthly = (
            met.groupby(['ScnID', 'comid', 'year', 'month'])['precip_mm']
            .sum()
            .reset_index()
            .rename(columns={'precip_mm': 'Pi_mm'})
        )

        # Annual totals
        annual = (
            met.groupby(['ScnID', 'comid', 'year'])['precip_mm']
            .sum()
            .reset_index()
            .rename(columns={'precip_mm': 'P_annual_mm'})
        )

        merged = monthly.merge(
            annual,
            on=['ScnID', 'comid', 'year'],
            how='left'
        )

        precip_records.append(merged)

    conn.close()

# %%
df_precip = pd.concat(precip_records, ignore_index=True)

In [ ]:
def compute_erosivity_R(monthly_df):
    """
    Input: monthly_df = 12-row dataframe for a single COMID-year
           with columns ['Pi_mm', 'P_annual_mm']
    Output: R value for that COMID-year
    """
    R_sum = 0
    
    p = monthly_df['P_annual_mm'].iloc[0]

    for _, row in monthly_df.iterrows():
        Pi = row['Pi_mm']

        # avoid divide-by-zero
        if p == 0 or Pi == 0:
            continue  

        term = 1.735 * 10 ** (1.5 * np.log10((Pi**2) / p) - 0.08188)
        R_sum += term

    return R_sum

In [ ]:
# Compute R for each COMID-year
R_per_year = (
    df_precip
    .groupby(['ScnID', 'comid', 'year'])
    .apply(compute_erosivity_R)
    .reset_index(name='R_annual')
)


# Compute mean annual R per COMID
mean_R_per_comid = (
    R_per_year
    .groupby(['ScnID', 'comid'])['R_annual']
    .mean()
    .reset_index(name='R_mean')
)

C:\Users\sbakar\AppData\Local\Temp\ipykernel_51812\4088926515.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_erosivity_R)


In [ ]:
# Create a dictionary mapping COMID → R_mean
R_dict = (
    mean_R_per_comid
    .set_index(['ScnID', 'comid'])['R_mean']
    .to_dict()
)

In [ ]:
# read in rasters for remaining factors K, C, and LS
rasterspath = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\GIS\2_USO_DE_SUELO_GEOLOGIA_LAND_USE_COVER\FACTORES_EROSION 2\FACTORES_EROSION"
# rasterspath = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\GIS\2_USO_DE_SUELO_GEOLOGIA_LAND_USE_COVER\FACTORES_EROSION 2\FACTORES_EROSION"
k_rasterpath = f"{rasterspath}/FactorK_250m.tif"
c_rasterpath = f"{rasterspath}/factorC_250m.tif"
ls_rasterpath = f"{rasterspath}/FactorLS_250m.tif"
p_rasterpath = f"{rasterspath}/FactorP_250m.tif"


In [ ]:
# ---- Zonal stats for K, C, LS, P ----
kcp_ls_stats = {}

for factor, path in zip(
        ['K_mean', 'C_mean', 'LS_mean', 'P_mean'],
        [k_rasterpath, c_rasterpath, ls_rasterpath, p_rasterpath]
    ):

    with rasterio.open(path) as src:
        raster_nodata = src.nodata

    stats = zonal_stats(
        subbasins_gdf,
        path,
        stats=['mean'],
        geojson_out=False,
        nodata=raster_nodata
    )

    vals = []
    for v in [s['mean'] for s in stats]:
        if v is None or (isinstance(v, (int, float)) and np.isinf(v)):
            vals.append(np.nan)
        else:
            vals.append(v)

    kcp_ls_stats[factor] = vals

# Add to GeoDataFrame
for factor in ['K_mean', 'C_mean', 'LS_mean', 'P_mean']:
    subbasins_gdf[factor] = kcp_ls_stats[factor]

rusle_records = []

for ScnID in mean_R_per_comid['ScnID'].unique():

    # Subset R values for this scenario
    R_scn = mean_R_per_comid.loc[
        mean_R_per_comid['ScnID'] == ScnID,
        ['comid', 'R_mean']
    ].copy()

    # Ensure COMID type consistency
    R_scn['comid'] = R_scn['comid'].astype(subbasins_gdf.index.dtype)

    # Temporary copy of subbasins
    gdf_tmp = subbasins_gdf.copy()

    # Assign R_mean
    gdf_tmp['R_mean'] = R_scn.set_index('comid')['R_mean']

    # Compute RUSLE A
    gdf_tmp['A_mean'] = (
        gdf_tmp['R_mean'] *
        gdf_tmp['K_mean'] *
        gdf_tmp['C_mean'] *
        gdf_tmp['LS_mean'] *
        gdf_tmp['P_mean']
    )

    df_tmp = gdf_tmp[
        ['R_mean', 'K_mean', 'C_mean', 'LS_mean', 'P_mean', 'A_mean']
    ].copy()

    df_tmp['ScnID'] = ScnID
    df_tmp['COMID'] = df_tmp.index

    rusle_records.append(df_tmp)

rusle_df = pd.concat(rusle_records, ignore_index=True)
print(rusle_df.tail())


        R_mean    K_mean    C_mean    LS_mean    P_mean     A_mean  ScnID  \
0  1347.914251  0.022797  0.226613   6.582823  0.583663  26.754705      1   
1   697.058802  0.021080  0.377415   2.687538  0.562313   8.380823      1   
2   951.112799  0.026806  0.495318   5.392525  0.593533  40.419098      1   
3     4.159786  0.025618  0.478399   2.108712  0.552899   0.059439      1   
4   945.428423  0.016664  0.123771  22.095901  0.701136  30.208947      1   

       COMID  
0  311153600  
1  310832400  
2  310825700  
3  310282400  
4  308754600  


In [26]:
# Initialize A_class with NaN
rusle_df['A_class'] = np.nan

# Mask for valid values
valid_mask = rusle_df['A_mean'].notna()

# Extract only valid A_mean values
a_values_valid = rusle_df.loc[valid_mask, 'A_mean'].values

# Compute 4 natural breaks on valid values only
classifier = mapclassify.NaturalBreaks(a_values_valid, k=4)

# Assign class labels (1–4) back to valid rows
rusle_df.loc[valid_mask, 'A_class'] = classifier.yb + 1

In [27]:
rusle_df = rusle_df.rename(columns={'COMID': 'comid'})  # Ensure COMID column is named correctly

In [37]:
#Update relevant dataframe and value column from the calculations above for the specific indicator
insert_data= rusle_df
value_column= 'A_mean'

In [38]:
# Connect to your SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Prepare list of rows to insert
rows_to_insert = []

#for scn_id, column_name in scenario_columns.items():
for _, row in insert_data.iterrows():
    scn_id = row['ScnID']
    comid = row['comid']
    value = row[value_column]

    rows_to_insert.append((scn_id, IndID, comid, value))

# Insert data into IndValues_Dyn
insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

In [39]:
# check that min and max values match the expected range based on the Indicators Table 
indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

print(f"Indicator {IndID}  Min: {ind_min}, Max: {ind_max}")

value_stats = (
    rusle_df['A_mean']
    .agg(['min', 'max', 'count'])
    .reset_index()
)

print("\n=== Values to be inserted (by scenario) ===")
print(value_stats)

# check for duplicates 
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("Duplicates in rows_to_insert:")
print(df_check[duplicates])

Indicator 108  Min: 1, Max: 4

=== Values to be inserted (by scenario) ===
   index       A_mean
0    min     0.000000
1    max  1278.126267
2  count  6523.000000
Duplicates in rows_to_insert:
Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [40]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()